In [1]:
#터미널에 pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

In [2]:
import torch
print(torch.cuda.is_available())        # True면 성공
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
#cmd -> 관리자권한실행 -> nvidia-smi 실행 -> 드라이버 설치되어있는지 ->없으면 엔비디아 확인 

In [4]:
print(torch.version.cuda)

13.2


In [5]:
import torch
print(torch.cuda.device_count())

1


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
iris = load_iris()
iris 

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

In [8]:
iris.target_names


array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [9]:
iris.target[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [10]:
#데이터분할 작업 
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
#8:2비율로 추출


In [11]:
scaler=StandardScaler()
X_train_tensor=torch.FloatTensor(scaler.fit_transform(X_train))
#텐서를 파이토치형으로 변환시켜줘야함 
X_test_tensor=torch.FloatTensor(scaler.transform(X_test))
y_train_tensor=torch.LongTensor(y_train)
y_test_tensor=torch.LongTensor(y_test)

In [12]:
class IrisNet(nn.Module):
    #신경망 클래스 만들때 기본모듈
    def __init__(self):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(4,16),
            nn.ReLU(),
            nn.Linear(16,3)
        )
    def forward(self,x):
        return self.layers(x)

In [13]:
model = IrisNet() #모델 객체 생성
criterion = nn.CrossEntropyLoss() #손실함수 정의
optimizer = optim.Adam(model.parameters(),lr=0.01) #최적화 함수

In [14]:
for _ in range(100):
    optimizer.zero_grad()
    loss=criterion(model(X_train_tensor),y_train_tensor)
    #손실값 계산
    loss.backward()
    #손실값에 따른 기울기(가중치 계산)
    optimizer.step()
    #기울기와 편향 수정

In [15]:
model.eval()
with torch.no_grad():
    pred=model(X_test_tensor).argmax(dim=1)
    acc=(pred==y_test_tensor).float().mean()
    print(f"테스트 정확도: {acc*100:.2f}%")
    for i in range(3):
        print(f"실제: {iris.target_names[y_test[i]]}, 예측: {iris.target_names[pred[i].item()]}")


테스트 정확도: 96.67%
실제: setosa, 예측: setosa
실제: virginica, 예측: virginica
실제: versicolor, 예측: versicolor


In [16]:
for _ in range(100):
    model.train()
    #학습모드로 전환 가중치없음 
    loss=criterion(model(X_train_tensor),y_train_tensor)
    #손실값 계산
    loss.backward()
    #손실값에 따른 기울기(가중치 계산)
    optimizer.step()
    #기울기와 편향 수정

In [17]:
model.eval() #평가모드로 전환 
with torch.no_grad(): #no_Grad() 가중치(기울기) 계산 비활성화
    pred=model(X_test_tensor).argmax(dim=1)
    acc=(pred==y_test_tensor).float().mean()
    print(f"테스트 정확도: {acc*100:.2f}%")
    for i in range(3):
        print(f"실제: {iris.target_names[y_test[i]]}, 예측: {iris.target_names[pred[i].item()]}")


테스트 정확도: 93.33%
실제: setosa, 예측: setosa
실제: virginica, 예측: virginica
실제: versicolor, 예측: versicolor


In [18]:
#보스턴집값

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader,TensorDataset

In [20]:
from sklearn.datasets import fetch_openml

boston=fetch_openml(name="boston",version=1,as_frame=True,parser="auto")
X=boston.data
y=boston.target

In [21]:
X.head(2)

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.9,4.98
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.9,9.14


In [22]:
y[:10]

0    24.0
1    21.6
2    34.7
3    33.4
4    36.2
5    28.7
6    22.9
7    27.1
8    16.5
9    18.9
Name: MEDV, dtype: float64

In [23]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [24]:
X_train,X_temp,y_train,y_temp = train_test_split(X_scaled,y,test_size=0.2,random_state=42)
X_val,X_test,y_val,y_test = train_test_split(X_temp,y_temp,test_size=0.5,random_state=42)

In [25]:
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train.values).view(-1,1)
X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val.values).view(-1,1)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test.values).view(-1,1)

In [26]:
from torch.utils.data import TensorDataset, DataLoader

#TensorDataset():입력데이터와 정답 데이터를 하나의 데이터셋으로 묶는다
train_dataset=TensorDataset(X_train,y_train)
#DataLoader():데이터를 모델에 전달하기 편리하도록 미니배치 단위로 나눈다.
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)

In [27]:
print(f"Train set:{X_train.shape}, Val set:{X_val.shape}, Test set:{X_test.shape}")


Train set:torch.Size([404, 13]), Val set:torch.Size([51, 13]), Test set:torch.Size([51, 13])


In [28]:
model = nn.Sequential(
    nn.Linear(13,64),
    nn.ReLU(),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,1)
)
print(model)

Sequential(
  (0): Linear(in_features=13, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


In [29]:
criterion=nn.MSELoss()
#MSELoss(): 회귀 (평균제곱오차, 집값처럼 연속된 숫자 예측)
#CrossEntropyLoss(): 다중분류 (모델 출력에 softmax 없이 그대로 넣음)
#BCELoss(): 이진분류 (모델 마지막에 Sigmoid를 넣어야 함, 0~1 확률값을 입력)
#BCEWithLogitsLoss(): 이진분류 (Sigmoid가 내장됨, 모델 출력을 그대로 넣음, BCELoss보다 계산이 안정적)

optimizer=optim.Adam(model.parameters(),lr=0.01)
#Adam: 가중치를 업데이트하는 최적화 함수
#model.parameters(): 학습할 가중치와 편향
#lr=0.01: 학습률 (한 번에 가중치를 얼마나 크게 바꿀지)

In [30]:
epochs=100
for epoch in range(epochs):
    model.train()
    for batch_X,batch_y in train_loader:
        optimizer.zero_grad()
        y_pred=model(batch_X)
        loss=criterion(y_pred,batch_y)
        loss.backward()
        optimizer.step()

    #검증
    model.eval()
    with torch.no_grad():
        val_pred=model(X_val)
        val_loss=criterion(val_pred,y_val)

    if (epoch+1)%10==0:
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

Epoch 10/100, Train Loss: 6.3930, Val Loss: 17.6370
Epoch 20/100, Train Loss: 6.3711, Val Loss: 17.0030
Epoch 30/100, Train Loss: 7.7085, Val Loss: 17.8708
Epoch 40/100, Train Loss: 3.1601, Val Loss: 18.4801
Epoch 50/100, Train Loss: 3.8355, Val Loss: 18.7477
Epoch 60/100, Train Loss: 5.2722, Val Loss: 17.6552
Epoch 70/100, Train Loss: 3.5867, Val Loss: 16.9318
Epoch 80/100, Train Loss: 8.8460, Val Loss: 16.6556
Epoch 90/100, Train Loss: 11.3127, Val Loss: 16.5420
Epoch 100/100, Train Loss: 5.2102, Val Loss: 15.9923


In [31]:
#model test
#테스트데이터로 평가
print("\n 테스트 세트로 모델 평가 중...")
model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    test_loss = criterion(test_pred,y_test)
    print(f"Test MSE Loss: {test_loss.item():.4f}")
    print(f"실제값(예시): {y_test[0].item():.2f}, 예측값: {test_pred[0].item():.2f}")

mae= torch.mean(torch.abs(test_pred-y_test))
print(f"Test MAE(오차평균): {mae.item():.4f}")


 테스트 세트로 모델 평가 중...
Test MSE Loss: 6.3555
실제값(예시): 23.10, 예측값: 23.00
Test MAE(오차평균): 1.8978


In [32]:
#GPU사용
#기존데이터를 gpu에서쓸수있도록 이동


In [ ]:
#모델객체를 만들고 to devic로 옮겨줌 
#쿠다 체크는 라이브러리 만들고 체크해줌 -> 데이터로드 ->데이터 전처리 ->모델객체 만들고 이동시킴 -> 손실함수

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader,TensorDataset

In [35]:
device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("사용 장치",device)

if torch.cuda.is_available():
    print("GPU이름:",torch.cuda.get_device_name(0))

사용 장치 cuda
GPU이름: NVIDIA GeForce RTX 5060 Laptop GPU


In [36]:
from sklearn.datasets import fetch_openml

boston=fetch_openml(name="boston",version=1,as_frame=True,parser="auto")
X=boston.data
y=boston.target

In [37]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [38]:
X_train,X_temp,y_train,y_temp = train_test_split(X_scaled,y,test_size=0.2,random_state=42)
X_val,X_test,y_val,y_test = train_test_split(X_temp,y_temp,test_size=0.5,random_state=42)

In [39]:
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train.values).view(-1,1)
X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val.values).view(-1,1)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test.values).view(-1,1)

In [40]:
from torch.utils.data import TensorDataset, DataLoader

#TensorDataset():입력데이터와 정답 데이터를 하나의 데이터셋으로 묶는다
train_dataset=TensorDataset(X_train,y_train)
#DataLoader():데이터를 모델에 전달하기 편리하도록 미니배치 단위로 나눈다.
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)

In [41]:
model = nn.Sequential(
    nn.Linear(13,64),
    nn.ReLU(),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,1)
).to(device)
#모델객체를 디바이스로옮겨줌
print(model)

Sequential(
  (0): Linear(in_features=13, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


In [42]:
criterion=nn.MSELoss()
#MSELoss(): 회귀 (평균제곱오차, 집값처럼 연속된 숫자 예측)
#CrossEntropyLoss(): 다중분류 (모델 출력에 softmax 없이 그대로 넣음)
#BCELoss(): 이진분류 (모델 마지막에 Sigmoid를 넣어야 함, 0~1 확률값을 입력)
#BCEWithLogitsLoss(): 이진분류 (Sigmoid가 내장됨, 모델 출력을 그대로 넣음, BCELoss보다 계산이 안정적)

optimizer=optim.Adam(model.parameters(),lr=0.01)
#Adam: 가중치를 업데이트하는 최적화 함수
#model.parameters(): 학습할 가중치와 편향
#lr=0.01: 학습률 (한 번에 가중치를 얼마나 크게 바꿀지)

In [45]:
model=model.to(device)


epochs=100
for epoch in range(epochs):
    model.train()
    for batch_X,batch_y in train_loader:
        batch_X=batch_X.to(device)
        batch_y=batch_y.to(device)

        optimizer.zero_grad()
        y_pred=model(batch_X)
        loss=criterion(y_pred,batch_y)
        loss.backward()
        optimizer.step()

    #검증
    model.eval()
    X_val=X_val.to(device)
    y_val=y_val.to(device)
    with torch.no_grad():
        val_pred=model(X_val)
        val_loss=criterion(val_pred,y_val)

    if (epoch+1)%10==0:
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

Epoch 10/100, Train Loss: 8.5261, Val Loss: 17.1402
Epoch 20/100, Train Loss: 4.6960, Val Loss: 17.9163
Epoch 30/100, Train Loss: 3.8107, Val Loss: 17.5465
Epoch 40/100, Train Loss: 12.6050, Val Loss: 17.9993
Epoch 50/100, Train Loss: 8.7613, Val Loss: 19.0273
Epoch 60/100, Train Loss: 5.7396, Val Loss: 18.4307
Epoch 70/100, Train Loss: 8.9555, Val Loss: 19.3441
Epoch 80/100, Train Loss: 6.7859, Val Loss: 19.3165
Epoch 90/100, Train Loss: 5.8182, Val Loss: 19.8357
Epoch 100/100, Train Loss: 13.0136, Val Loss: 18.6835


In [47]:
#model test
#테스트데이터로 평가
print("\n 테스트 세트로 모델 평가 중...")
X_test=X_test.to(device)
y_test=y_test.to(device)
model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    test_loss = criterion(test_pred,y_test)
    print(f"Test MSE Loss: {test_loss.item():.4f}")
    print(f"실제값(예시): {y_test[0].item():.2f}, 예측값: {test_pred[0].item():.2f}")

mae= torch.mean(torch.abs(test_pred-y_test))
print(f"Test MAE(오차평균): {mae.item():.4f}")


 테스트 세트로 모델 평가 중...
Test MSE Loss: 7.7761
실제값(예시): 23.10, 예측값: 21.91
Test MAE(오차평균): 2.0811
